# Combined Agent: UC-First + Genie Fallback + Lakebase Memory + Parallel MCP

This notebook combines the best features from both agent architectures:

## Architecture

```
User Query → UC Functions (Parallel MCP) → Sufficiency Check → [If sufficient] → Response
                                                             ↓
                                                     [If partial/insufficient]
                                                             ↓
                                              Genie Fallback (unanswered parts only)
                                                             ↓
                                                   Synthesize → Response
```

## Features

### From UC-First Genie-Fallback:
- **UC functions tried FIRST** - deterministic, fast
- **Partial answer detection** - identifies what was/wasn't answered
- **Targeted Genie queries** - only asks about unanswered parts
- **Intelligent synthesis** - combines both responses seamlessly

### From Lakebase + MCP Agent:
- **Parallel tool execution** via MCP - all tools run simultaneously
- **Lakebase PostgreSQL memory** - conversation persistence across sessions
- **Connection pooling** - efficient database connections
- **OAuth credential caching** - 50-minute token cache

## Prerequisites

1. Run `00_setup` notebook first to create `config/atbat_assistant.json`
2. Ensure `genie` and `lakebase` sections are in the config
3. Lakebase instance configured (for memory)

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" databricks-mcp langgraph-checkpoint-postgres psycopg[binary,pool] databricks-langchain langgraph
dbutils.library.restartPython()

## Load Configuration

Load configuration from `config/atbat_assistant.json` (created by setup notebook).

In [ ]:
# Load configuration from setup notebook
import json
import os
from pathlib import Path
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Extract configuration variables
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
FINAL_SYNTHESIS_ENDPOINT_NAME = os.getenv(
    "ATBAT_FINAL_SYNTHESIS_ENDPOINT",
    CONFIG["llm"].get("final_synthesis_endpoint_name", "gpt-5-4-external"),
)
FINAL_SYNTHESIS_ENDPOINT_NAME = str(FINAL_SYNTHESIS_ENDPOINT_NAME or "").replace("databricks:/", "")
UC_MODEL_NAME = CONFIG["model"]["uc_model_name"]
MODEL_NAME = CONFIG["model"]["model_name"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
DISABLE_VECTOR_TOOLS = os.getenv("DISABLE_VECTOR_TOOLS", "true").lower() in {"1", "true", "yes"}
if DISABLE_VECTOR_TOOLS:
    UC_TOOL_NAMES = [
        tool_name for tool_name in UC_TOOL_NAMES
        if "embedding" not in tool_name.lower() and "vector" not in tool_name.lower()
    ]
    print(f"[CONFIG] Vector and embedding tools disabled; using {len(UC_TOOL_NAMES)} UC tools")
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]

# Genie config
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]

# Lakebase config
LAKEBASE_INSTANCE = CONFIG["lakebase"]["instance_name"]
LAKEBASE_HOST = CONFIG["lakebase"].get("conn_host") or CONFIG["lakebase"].get("host") or ""

# Set MLflow experiment
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

print(f"Loaded config from: config/atbat_assistant.json")
print(f"LLM Endpoint: {LLM_ENDPOINT_NAME}")
print(f"Final synthesis endpoint: {FINAL_SYNTHESIS_ENDPOINT_NAME or '(runtime model)'}")
print(f"UC Tools: {len(UC_TOOL_NAMES)} functions")
print(f"Genie Space: {GENIE_SPACE_ID}")
print(f"Lakebase Host: {LAKEBASE_HOST or '(not configured - memory disabled)'}")


## Register System Prompt

The prompt includes routing logic that prioritizes UC Functions for precise tasks
and falls back to the Genie Space for broader data exploration.

In [ ]:
# Define and register prompts used by the runtime agent.
PROMPT_TEMPLATE = """You are a hitting assistant tasked with helping batters prepare for matchups against specific pitchers.

Use deterministic matchup history, pitcher tendency, arsenal, roster, lineup, and Genie-backed table analysis to answer questions.

The team abbreviations to choose from are below, use the 3 letter acronyms to get data:

Teams:TEX,CHC,LAA,LAD,STL,PHI,ARI,OAK,TBR,MIN,CLE,CHW,NYM,COL,SEA,MIA,SDP,WSN,HOU,SFG,CIN,BAL,KCR,PIT,ATL,NYY,DET,MIL,TOR,BOS,ATH

General rules:

- Always assume the most recent season (2025) if a season is not provided.
- Always leverage the tooling you have available to answer user queries.
- Only perform the minimum necessary tool calls to complete a request. Do not exceed 8 tool calls before providing a response.
- If you need multiple tools, include all tool_calls in a single assistant message; don't chain them one-by-one.
- Before calling a tool, include every required argument. Default season_year to 2025 when missing. For runner-state tendency tools, set unspecified base occupancy flags to false and use a 0-0 count when the user did not specify a count.
- If a tool times out, returns a 504, or returns empty rows, do not end with an apology. Use alternate available evidence when possible and provide a next-best coaching plan.

For open-ended requests always respond with the following format in markdown:
# At-Bat Assistant Assessment
## Data collected
- Summarize the data collected to inform the analysis in a very concise fashion. You do not need reference the tools by their explicit name, just need to summarize the data collected. Ex. Collected data on tendencies by count. Do not exceed 50 words
## Pitcher Approach
- Summarize how the pitcher might approach the batter. Use discretion to include further subheadings by count or scenario, or just include it all under the pitcher approach heading if that is not necessary. Do not exceed 200 words
## Recommendation
- Summarize how the batter should approach potential at-bats(s) in a concise format, 50-75 words.

Fallback rules:
- If exact data is unavailable, still provide a grounded action plan with: what was checked, what was unavailable, and how the hitter should adjust using pitch mix, handedness, count, location, and runner-state principles.
- Do not claim the baseball data does not exist unless all available retrieval paths have failed or returned no rows.

CONVERSATION HISTORY: You have access to previous messages in this conversation thread.
- ONLY reference prior conversation context if the user's current question is ambiguous or explicitly refers to something discussed earlier.
- If the user asks a NEW, self-contained question, answer it directly using your tools WITHOUT referencing prior context.
- Do NOT proactively bring up previous topics or assume the user wants comparisons to earlier queries.
"""

SUFFICIENCY_PROMPT_NAME = f"{PROMPT_NAME}_sufficiency"
SYNTHESIS_PROMPT_NAME = f"{PROMPT_NAME}_synthesis"
PARTIAL_SYNTHESIS_PROMPT_NAME = f"{PROMPT_NAME}_partial_synthesis"

SUFFICIENCY_PROMPT_TEMPLATE = """You are evaluating whether a response adequately answers a user's question.
The question may have MULTIPLE parts. Evaluate EACH part separately.

Original Question: {original_query}

Draft Response from UC Functions Agent:
{uc_response}

Observed Tool Calls and Outputs:
{tool_summary}

Evaluation rules:
- Judge the drafted response, not whether tools were merely called.
- Use the tool outputs as supporting evidence to decide whether the drafted response answered the question.
- FULLY_ANSWERED means the response addresses every material part of the question, even if the answer is that no relevant records exist.
- PARTIALLY_ANSWERED means the response answers only some parts, omits a requested recommendation, or ignores relevant evidence.
- NOT_ANSWERED means the response is empty, only reports an execution failure, or does not materially address the user's request.
- If the tool outputs contain relevant evidence that the response failed to use, prefer PARTIALLY_ANSWERED over FULLY_ANSWERED.

Respond in this EXACT format:
STATUS: [FULLY_ANSWERED or PARTIALLY_ANSWERED or NOT_ANSWERED]
ANSWERED_PARTS: [Brief summary of what was answered, or "None"]
UNANSWERED_PARTS: [Specific questions that still need answers, or "None"]
RATIONALE: [One brief sentence]

Your evaluation:"""

PARTIAL_SYNTHESIS_PROMPT_TEMPLATE = """Combine two partial responses into a unified answer.

CURRENT QUESTION (answer ONLY this): {original_query}

PART 1 - From UC Functions:
{uc_response}

PART 2 - From Genie:
{genie_response}

IMPORTANT: Only include information that DIRECTLY answers the current question above.
Do NOT include information from prior conversation turns unless explicitly referenced.
Create a UNIFIED response that presents all relevant information clearly without mentioning different systems."""

SYNTHESIS_PROMPT_TEMPLATE = """Synthesize a response based on available information.

CURRENT QUESTION (answer ONLY this): {original_query}

Available Information:
- UC Functions: {uc_response}
- Genie: {genie_response}

IMPORTANT: Only include information that DIRECTLY answers the current question.
Provide a clear, focused response."""


def _register_prompt_if_changed(prompt_name, template, initial_message, update_message):
    try:
        existing_prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@production")
        if existing_prompt.template == template:
            print(f"Prompt '{prompt_name}' is up-to-date (version {existing_prompt.version}, @production)")
            return existing_prompt
        print(f"Prompt '{prompt_name}' template has changed. Registering new version...")
        prompt = mlflow.genai.register_prompt(
            name=prompt_name,
            template=template,
            commit_message=update_message,
        )
    except Exception:
        print(f"Prompt '{prompt_name}' not found. Registering...")
        prompt = mlflow.genai.register_prompt(
            name=prompt_name,
            template=template,
            commit_message=initial_message,
        )

    mlflow.genai.set_prompt_alias(name=prompt_name, alias="production", version=prompt.version)
    print(f"Set {prompt_name}@production -> version {prompt.version}")
    return prompt


system_prompt = _register_prompt_if_changed(
    PROMPT_NAME,
    PROMPT_TEMPLATE,
    "Initial no-embeddings at-bat assistant prompt",
    "Update baseline prompt to no-embeddings runtime behavior",
)
sufficiency_prompt = _register_prompt_if_changed(
    SUFFICIENCY_PROMPT_NAME,
    SUFFICIENCY_PROMPT_TEMPLATE,
    "Initial sufficiency evaluation prompt",
    "Update sufficiency evaluation prompt",
)
synthesis_prompt = _register_prompt_if_changed(
    SYNTHESIS_PROMPT_NAME,
    SYNTHESIS_PROMPT_TEMPLATE,
    "Initial final synthesis prompt",
    "Update final synthesis prompt",
)
partial_synthesis_prompt = _register_prompt_if_changed(
    PARTIAL_SYNTHESIS_PROMPT_NAME,
    PARTIAL_SYNTHESIS_PROMPT_TEMPLATE,
    "Initial partial synthesis prompt",
    "Update partial synthesis prompt",
)


## Setup Lakebase (Optional)

If you want conversation memory, set up Lakebase. Skip this cell if you don't need memory.

In [ ]:
# Only run if you have Lakebase configured and need to set up checkpoint tables
SETUP_LAKEBASE = False  # Set to True to create checkpoint tables
CHECKPOINT_SCHEMA = "checkpoint"  # Custom schema we own

if SETUP_LAKEBASE:
    from langgraph.checkpoint.postgres import PostgresSaver
    import psycopg
    import uuid

    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()

    current_user = w.current_user.me().user_name

    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()),
        instance_names=[LAKEBASE_INSTANCE],
    )

    conn = psycopg.connect(
        f"dbname=databricks_postgres user={current_user} host={LAKEBASE_HOST} sslmode=require",
        password=cred.token
    )
    conn.autocommit = True

    # Create a schema we own
    conn.execute(f'CREATE SCHEMA IF NOT EXISTS {CHECKPOINT_SCHEMA}')
    conn.execute(f'SET search_path TO {CHECKPOINT_SCHEMA}')
    print(f"Created and switched to schema: {CHECKPOINT_SCHEMA}")

    # Grant SP privileges on the new schema
    SP_CLIENT_ID = dbutils.secrets.get(scope=CONFIG["prompt_registry_auth"]["secret_scope_name"],
                                        key=CONFIG["prompt_registry_auth"]["oauth_client_id_key"])
    conn.execute(f'GRANT ALL ON SCHEMA {CHECKPOINT_SCHEMA} TO "{SP_CLIENT_ID}"')
    print(f"Granted schema privileges to SP: {SP_CLIENT_ID}")

    checkpointer = PostgresSaver(conn)
    checkpointer.setup()
    print("Lakebase checkpoint tables created!")

    # Grant SP access to the checkpoint tables
    conn.execute(f'GRANT ALL ON ALL TABLES IN SCHEMA {CHECKPOINT_SCHEMA} TO "{SP_CLIENT_ID}"')
    print(f"Granted table privileges to SP")

    conn.close()
    print(f"\nIMPORTANT: The agent's Lakebase connection must use search_path={CHECKPOINT_SCHEMA}")
else:
    print("Skipping Lakebase setup (set SETUP_LAKEBASE = False to create checkpoint tables)")

## Define the Agent Code

Below we define the combined agent code in a single cell, enabling us to write it to a local Python file using the `%%writefile` magic command for subsequent logging and deployment.

### Agent Features:
- **UC-first, Genie-fallback architecture** - Deterministic tools tried first
- **Parallel MCP tool execution** - All tool calls run simultaneously  
- **Partial answer detection** - Identifies what was/wasn't answered
- **Lakebase memory** - PostgreSQL-backed conversation persistence
- **Connection pooling** - Efficient database connections with OAuth caching

In [ ]:
%%writefile agent.py
"""
Combined Agent: UC-First with Genie Fallback + Lakebase Memory + Parallel MCP Tool Calling

This agent combines:
1. UC functions tried FIRST via parallel MCP execution (deterministic, fast)
2. Sufficiency evaluation with partial answer detection
3. Genie fallback for unanswered parts (flexible, novel queries)
4. Lakebase PostgreSQL memory for conversation persistence
5. Connection pooling and OAuth credential caching
"""

import asyncio
import json
import logging
import os
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from pathlib import Path
from threading import Lock
from typing import Annotated, Any, Generator, Literal, Optional, Sequence, TypedDict

import mlflow
import psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.config import Config
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from databricks_langchain.genie import GenieAgent
from databricks_mcp import DatabricksMCPClient
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, ToolMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
from psycopg.rows import dict_row
from psycopg_pool import ConnectionPool
from pydantic import BaseModel

logger = logging.getLogger(__name__)

########################################
# UTILITIES
########################################

def get_dbutils():
    """Get dbutils for secrets access."""
    try:
        from pyspark.dbutils import DBUtils
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        return DBUtils(spark)
    except (ImportError, Exception):
        try:
            import IPython
            ipython = IPython.get_ipython()
            if ipython and "dbutils" in ipython.user_ns:
                return ipython.user_ns["dbutils"]
        except:
            pass
    return None

dbutils = get_dbutils()

########################################
# CONFIGURATION
########################################

_CONFIG_PATH = Path("config/atbat_assistant.json")
if _CONFIG_PATH.exists():
    CONFIG = json.loads(_CONFIG_PATH.read_text())
else:
    config_env = os.getenv("ATBAT_ASSISTANT_CONFIG_JSON")
    if config_env:
        CONFIG = json.loads(config_env)
    else:
        raise FileNotFoundError(
            "config/atbat_assistant.json not found and ATBAT_ASSISTANT_CONFIG_JSON env var is not set"
        )

# Extract configuration values
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
DISABLE_VECTOR_TOOLS = os.getenv("DISABLE_VECTOR_TOOLS", "true").lower() in {"1", "true", "yes"}
if DISABLE_VECTOR_TOOLS:
    UC_TOOL_NAMES = [
        tool_name for tool_name in UC_TOOL_NAMES
        if "embedding" not in tool_name.lower() and "vector" not in tool_name.lower()
    ]
    print(f"[CONFIG] Vector and embedding tools disabled; using {len(UC_TOOL_NAMES)} UC tools")
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]

# Genie configuration
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]
SQL_WAREHOUSE_ID = CONFIG["genie"].get("warehouse_id") or os.getenv("DATABRICKS_SQL_WAREHOUSE_ID", "")

# Auth configuration
SECRET_SCOPE_NAME = CONFIG["prompt_registry_auth"]["secret_scope_name"]
CLIENT_ID_KEY = CONFIG["prompt_registry_auth"]["oauth_client_id_key"]
CLIENT_SECRET_KEY = CONFIG["prompt_registry_auth"]["oauth_client_secret_key"]
DATABRICKS_HOST = CONFIG["prompt_registry_auth"]["databricks_host"]

# Set DATABRICKS_HOST if configured
if DATABRICKS_HOST and not os.getenv("DATABRICKS_HOST"):
    os.environ["DATABRICKS_HOST"] = DATABRICKS_HOST.rstrip("/")

# Load OAuth credentials from secrets into LOCAL variables (not env vars).
# The notebook runtime already sets DATABRICKS_TOKEN in the environment;
# putting OAuth creds there too causes a dual-auth conflict in the SDK.
_SP_CLIENT_ID = None
_SP_CLIENT_SECRET = None
if dbutils:
    try:
        _SP_CLIENT_ID = dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=CLIENT_ID_KEY).strip()
        _SP_CLIENT_SECRET = dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=CLIENT_SECRET_KEY).strip()
    except:
        pass

# Create WorkspaceClient: use OAuth M2M if available, otherwise auto-detect from env.
# Explicit auth_type prevents the SDK from discovering DATABRICKS_TOKEN in the
# environment and raising a dual-auth conflict.
if _SP_CLIENT_ID and _SP_CLIENT_SECRET:
    WORKSPACE_CLIENT = WorkspaceClient(config=Config(
        host=os.environ.get("DATABRICKS_HOST"),
        client_id=_SP_CLIENT_ID,
        client_secret=_SP_CLIENT_SECRET,
        auth_type="oauth-m2m",
    ))
else:
    WORKSPACE_CLIENT = WorkspaceClient()

# Ensure DATABRICKS_HOST is set
if not os.environ.get("DATABRICKS_HOST") and WORKSPACE_CLIENT.config.host:
    os.environ["DATABRICKS_HOST"] = WORKSPACE_CLIENT.config.host.rstrip("/")

# MLflow setup
if os.environ.get("DATABRICKS_HOST") and not os.environ.get("MLFLOW_TRACKING_URI"):
    os.environ["MLFLOW_TRACKING_URI"] = "databricks"

mlflow.set_registry_uri("databricks-uc")
MLFLOW_EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
if MLFLOW_EXPERIMENT_ID:
    try:
        mlflow.set_experiment(experiment_id=MLFLOW_EXPERIMENT_ID)
        logger.info(f"MLflow experiment set to {MLFLOW_EXPERIMENT_ID}")
    except Exception as e:
        logger.warning(f"Could not set MLflow experiment: {e}")

PROMPT_TEMPLATE_FALLBACK = """You are a hitting assistant tasked with helping batters prepare for matchups against specific pitchers.

Use deterministic matchup history, pitcher tendency, arsenal, roster, lineup, and Genie-backed table analysis to answer questions.

The team abbreviations to choose from are below, use the 3 letter acronyms to get data:

Teams:TEX,CHC,LAA,LAD,STL,PHI,ARI,OAK,TBR,MIN,CLE,CHW,NYM,COL,SEA,MIA,SDP,WSN,HOU,SFG,CIN,BAL,KCR,PIT,ATL,NYY,DET,MIL,TOR,BOS,ATH

General rules:

- Always assume the most recent season (2025) if a season is not provided.
- Always leverage the tooling you have available to answer user queries.
- Only perform the minimum necessary tool calls to complete a request. Do not exceed 8 tool calls before providing a response.
- If you need multiple tools, include all tool_calls in a single assistant message; don't chain them one-by-one.
- Before calling a tool, include every required argument. Default season_year to 2025 when missing. For runner-state tendency tools, set unspecified base occupancy flags to false and use a 0-0 count when the user did not specify a count.
- If a tool times out, returns a 504, or returns empty rows, do not end with an apology. Use alternate available evidence when possible and provide a next-best coaching plan.

For open-ended requests always respond with the following format in markdown:
# At-Bat Assistant Assessment
## Data collected
- Summarize the data collected to inform the analysis in a very concise fashion. You do not need reference the tools by their explicit name, just need to summarize the data collected. Ex. Collected data on tendencies by count. Do not exceed 50 words
## Pitcher Approach
- Summarize how the pitcher might approach the batter. Use discretion to include further subheadings by count or scenario, or just include it all under the pitcher approach heading if that is not necessary. Do not exceed 200 words
## Recommendation
- Summarize how the batter should approach potential at-bats(s) in a concise format, 50-75 words.

Fallback rules:
- If exact data is unavailable, still provide a grounded action plan with: what was checked, what was unavailable, and how the hitter should adjust using pitch mix, handedness, count, location, and runner-state principles.
- Do not claim the baseball data does not exist unless all available retrieval paths have failed or returned no rows.

CONVERSATION HISTORY: You have access to previous messages in this conversation thread.
- ONLY reference prior conversation context if the user's current question is ambiguous or explicitly refers to something discussed earlier.
- If the user asks a NEW, self-contained question, answer it directly using your tools WITHOUT referencing prior context.
- Do NOT proactively bring up previous topics or assume the user wants comparisons to earlier queries.
"""

SUFFICIENCY_PROMPT_NAME = f"{PROMPT_NAME}_sufficiency"
SYNTHESIS_PROMPT_NAME = f"{PROMPT_NAME}_synthesis"
PARTIAL_SYNTHESIS_PROMPT_NAME = f"{PROMPT_NAME}_partial_synthesis"

SUFFICIENCY_PROMPT_FALLBACK = """You are evaluating whether a response adequately answers a user's question.
The question may have MULTIPLE parts. Evaluate EACH part separately.

Original Question: {original_query}

Draft Response from UC Functions Agent:
{uc_response}

Observed Tool Calls and Outputs:
{tool_summary}

Evaluation rules:
- Judge the drafted response, not whether tools were merely called.
- Use the tool outputs as supporting evidence to decide whether the drafted response answered the question.
- FULLY_ANSWERED means the response addresses every material part of the question, even if the answer is that no relevant records exist.
- PARTIALLY_ANSWERED means the response answers only some parts, omits a requested recommendation, or ignores relevant evidence.
- NOT_ANSWERED means the response is empty, only reports an execution failure, or does not materially address the user's request.
- If the tool outputs contain relevant evidence that the response failed to use, prefer PARTIALLY_ANSWERED over FULLY_ANSWERED.

Respond in this EXACT format:
STATUS: [FULLY_ANSWERED or PARTIALLY_ANSWERED or NOT_ANSWERED]
ANSWERED_PARTS: [Brief summary of what was answered, or "None"]
UNANSWERED_PARTS: [Specific questions that still need answers, or "None"]
RATIONALE: [One brief sentence]

Your evaluation:"""

PARTIAL_SYNTHESIS_PROMPT_FALLBACK = """Combine two partial responses into a unified answer.

CURRENT QUESTION (answer ONLY this): {original_query}

PART 1 - From UC Functions:
{uc_response}

PART 2 - From Genie:
{genie_response}

IMPORTANT: Only include information that DIRECTLY answers the current question above.
Do NOT include information from prior conversation turns unless explicitly referenced.
Create a UNIFIED response that presents all relevant information clearly without mentioning different systems."""

SYNTHESIS_PROMPT_FALLBACK = """Synthesize a response based on available information.

CURRENT QUESTION (answer ONLY this): {original_query}

Available Information:
- UC Functions: {uc_response}
- Genie: {genie_response}

IMPORTANT: Only include information that DIRECTLY answers the current question.
Provide a clear, focused response."""


def _load_prompt_or_fallback(prompt_name: str, fallback_template: str, label: str):
    try:
        return mlflow.genai.load_prompt(f"prompts:/{prompt_name}@production")
    except Exception as e:
        logger.warning(f"Could not load {label} prompt from registry, using embedded fallback: {e}")
        return fallback_template


def _format_prompt_template(prompt_obj: Any, **kwargs) -> str:
    template = getattr(prompt_obj, "template", None)
    if template is not None:
        return str(template).format(**kwargs)
    if hasattr(prompt_obj, "format"):
        try:
            return prompt_obj.format(**kwargs)
        except TypeError:
            return prompt_obj.format().format(**kwargs)
    return str(prompt_obj).format(**kwargs)


SYSTEM_PROMPT = _load_prompt_or_fallback(PROMPT_NAME, PROMPT_TEMPLATE_FALLBACK, "system")
SUFFICIENCY_PROMPT = _load_prompt_or_fallback(SUFFICIENCY_PROMPT_NAME, SUFFICIENCY_PROMPT_FALLBACK, "sufficiency")
SYNTHESIS_PROMPT = _load_prompt_or_fallback(SYNTHESIS_PROMPT_NAME, SYNTHESIS_PROMPT_FALLBACK, "synthesis")
PARTIAL_SYNTHESIS_PROMPT = _load_prompt_or_fallback(PARTIAL_SYNTHESIS_PROMPT_NAME, PARTIAL_SYNTHESIS_PROMPT_FALLBACK, "partial synthesis")

# Lakebase Configuration
DISABLE_LAKEBASE = os.getenv("DISABLE_LAKEBASE", "true").lower() == "true"

if DISABLE_LAKEBASE:
    print("[CONFIG] Lakebase DISABLED via DISABLE_LAKEBASE env var")
    LAKEBASE_CONFIG = {"instance_name": "", "conn_host": ""}
else:
    LAKEBASE_CONFIG = {
        "instance_name": CONFIG["lakebase"]["instance_name"],
        "conn_host": CONFIG["lakebase"].get("conn_host") or CONFIG["lakebase"].get("host") or "",
        "conn_db_name": "databricks_postgres",
        "conn_ssl_mode": "require",
    }
    print(f"[CONFIG] Lakebase instance: {LAKEBASE_CONFIG['instance_name']}")
    print(f"[CONFIG] Lakebase host: {LAKEBASE_CONFIG['conn_host'] or '(not configured)'}")

def _content_to_text(content: Any) -> str:
    """Convert LangChain/OpenAI content blocks to user-visible text."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                if item.get("type") == "reasoning":
                    continue
                parts.append(str(item.get("text") or item.get("content") or ""))
            elif hasattr(item, "text"):
                parts.append(str(item.text))
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)


def _strip_reasoning_artifacts(content: Any) -> str:
    """Remove GPT OSS reasoning blocks that can leak into text content."""
    text = _content_to_text(content)
    decoder = json.JSONDecoder()
    cleaned = []
    remaining = text

    while remaining:
        stripped = remaining.lstrip()
        if stripped.startswith("["):
            try:
                obj, end = decoder.raw_decode(stripped)
                if isinstance(obj, list) and obj and isinstance(obj[0], dict) and obj[0].get("type") == "reasoning":
                    remaining = stripped[end:]
                    continue
            except Exception:
                pass

        cleaned.append(remaining[0])
        remaining = remaining[1:]

    return "".join(cleaned).strip()

########################################
# MCP PARALLEL EXECUTION
########################################

# Cache auth for per-thread MCP clients (reuse the same local SP creds)
_CACHED_HOST = os.environ.get("DATABRICKS_HOST")
_CACHED_TOKEN = os.environ.get("DATABRICKS_TOKEN")
_FUNCTION_INFO_CACHE: dict[str, Any] = {}


def _sql_literal(value: Any) -> str:
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    escaped = str(value).replace("'", "''")
    return f"'{escaped}'"


def _get_sql_warehouse_id() -> str:
    if SQL_WAREHOUSE_ID:
        return SQL_WAREHOUSE_ID
    warehouses = list(WORKSPACE_CLIENT.warehouses.list())
    for warehouse in warehouses:
        if getattr(warehouse, "enable_serverless_compute", False):
            return warehouse.id
    if warehouses:
        return warehouses[0].id
    raise RuntimeError("No SQL warehouse available for UC function execution")


def _get_function_info(full_name: str):
    if full_name not in _FUNCTION_INFO_CACHE:
        _FUNCTION_INFO_CACHE[full_name] = WORKSPACE_CLIENT.functions.get(full_name)
    return _FUNCTION_INFO_CACHE[full_name]


_DEFAULT_TOOL_ARGS = {
    "season_year": 2025,
    "b": 0,
    "s": 0,
    "p_on_1b": False,
    "p_on_2b": False,
    "p_on_3b": False,
}

_TRANSIENT_TOOL_ERROR_MARKERS = (
    "504 gateway timeout",
    "execution timed out after 90 seconds",
    "provisioning resources for function execution",
    "temporarily unavailable",
    "timed out",
    "read timed out",
)


def _is_transient_tool_error(exc: BaseException) -> bool:
    text = str(exc).lower()
    return any(marker in text for marker in _TRANSIENT_TOOL_ERROR_MARKERS)


def _normalize_tool_args(tool_name: str, args: dict | None) -> tuple[dict, list[str]]:
    """Fill safe defaults for known UC tool arguments and report missing required args."""
    normalized = dict(args or {})
    mcp_tool_name = _resolve_tool_name(tool_name.replace(".", "__") if "." in tool_name else tool_name)
    full_name = mcp_tool_name.replace("__", ".")

    try:
        function_info = _get_function_info(full_name)
        parameters = list(getattr(getattr(function_info, "input_params", None), "parameters", []) or [])
    except Exception:
        parameters = []

    missing_required = []
    for param in sorted(parameters, key=lambda p: p.position or 0):
        name = param.name
        if name in normalized and normalized[name] is not None:
            continue
        if name in _DEFAULT_TOOL_ARGS:
            normalized[name] = _DEFAULT_TOOL_ARGS[name]
        else:
            missing_required.append(name)

    return normalized, missing_required


def _execute_uc_function_with_retries(tool_name: str, args: dict | None) -> str:
    """Validate UC tool args and retry transient Databricks execution failures."""
    normalized_args, missing_required = _normalize_tool_args(tool_name, args)
    if missing_required:
        missing = ", ".join(missing_required)
        raise ValueError(f"Missing required tool parameter(s) for {tool_name}: {missing}")

    max_attempts = int(os.getenv("UC_TOOL_MAX_ATTEMPTS", "2"))
    base_sleep = float(os.getenv("UC_TOOL_RETRY_BASE_SLEEP_SECONDS", "1.5"))
    last_error = None
    for attempt in range(max_attempts):
        try:
            return _execute_uc_function_mcp(tool_name, normalized_args)
        except Exception as exc:
            last_error = exc
            if attempt >= max_attempts - 1 or not _is_transient_tool_error(exc):
                raise
            sleep_seconds = base_sleep * (2 ** attempt)
            logger.warning(
                f"Retrying transient UC tool failure for {tool_name} "
                f"in {sleep_seconds:.1f}s ({attempt + 1}/{max_attempts}): {exc}"
            )
            time.sleep(sleep_seconds)

    raise last_error or RuntimeError(f"UC tool execution failed for {tool_name}")


def _format_statement_result(resp: Any) -> str:
    columns = []
    try:
        columns = [col.name for col in resp.manifest.schema.columns]
    except Exception:
        pass

    rows = []
    try:
        rows = resp.result.data_array or []
    except Exception:
        rows = []

    truncated = False
    try:
        truncated = bool(resp.manifest.truncated)
    except Exception:
        pass

    return json.dumps({
        "is_truncated": truncated,
        "columns": columns,
        "rows": rows,
    })


def _execute_uc_function_sql(tool_name: str, args: dict) -> str:
    """Execute a UC function through SQL Statement Execution.

    MCP function execution can hang on Free Edition serverless. SQL execution is
    deterministic for these UC functions and gives us statement-level polling.
    """
    mcp_tool_name = _resolve_tool_name(tool_name.replace(".", "__") if "." in tool_name else tool_name)
    full_name = mcp_tool_name.replace("__", ".")
    function_info = _get_function_info(full_name)
    parameters = list(getattr(getattr(function_info, "input_params", None), "parameters", []) or [])
    ordered_names = [p.name for p in sorted(parameters, key=lambda p: p.position or 0)]

    named_args = []
    for name in ordered_names:
        if name in args:
            named_args.append(f"{name} => {_sql_literal(args[name])}")
    if not named_args:
        for name, value in args.items():
            named_args.append(f"{name} => {_sql_literal(value)}")

    invocation = f"{full_name}({', '.join(named_args)})"
    data_type = str(getattr(function_info, "data_type", "")).upper()
    if "TABLE_TYPE" in data_type:
        statement = f"SELECT * FROM {invocation} LIMIT 100"
    else:
        statement = f"SELECT {invocation} AS output"

    timeout_seconds = int(os.getenv("UC_SQL_TOOL_TIMEOUT_SECONDS", "120"))
    deadline = time.time() + timeout_seconds
    resp = WORKSPACE_CLIENT.statement_execution.execute_statement(
        warehouse_id=_get_sql_warehouse_id(),
        statement=statement,
        wait_timeout="30s",
    )

    while str(resp.status.state).endswith("PENDING") or str(resp.status.state).endswith("RUNNING"):
        if time.time() >= deadline:
            try:
                WORKSPACE_CLIENT.statement_execution.cancel_execution(resp.statement_id)
            except Exception:
                pass
            raise TimeoutError(f"SQL function execution timed out after {timeout_seconds}s: {full_name}")
        time.sleep(2)
        resp = WORKSPACE_CLIENT.statement_execution.get_statement(resp.statement_id)

    state = str(resp.status.state)
    if not state.endswith("SUCCEEDED"):
        error = getattr(resp.status, "error", None)
        raise RuntimeError(f"SQL function execution failed for {full_name}: {state} {error}")

    return _format_statement_result(resp)


def _execute_uc_function_mcp(tool_name: str, args: dict) -> str:
    """Execute a UC function using Databricks MCP with per-thread client isolation."""
    if os.getenv("USE_MCP_FUNCTION_EXECUTION", "false").lower() not in {"1", "true", "yes"}:
        return _execute_uc_function_sql(tool_name, args)

    host = os.environ.get("DATABRICKS_HOST", "").rstrip("/")
    if not host:
        host = WORKSPACE_CLIENT.config.host.rstrip("/")
    mcp_server_url = f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"

    mcp_tool_name = _resolve_tool_name(tool_name.replace(".", "__") if "." in tool_name else tool_name)

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        # Create per-thread client. Explicit auth_type avoids dual-auth.
        # conflicts when the notebook runtime re-injects DATABRICKS_TOKEN.
        if _SP_CLIENT_ID and _SP_CLIENT_SECRET:
            thread_workspace_client = WorkspaceClient(config=Config(
                host=_CACHED_HOST,
                client_id=_SP_CLIENT_ID,
                client_secret=_SP_CLIENT_SECRET,
                auth_type="oauth-m2m",
            ))
        elif _CACHED_TOKEN:
            thread_workspace_client = WorkspaceClient(config=Config(
                host=_CACHED_HOST,
                token=_CACHED_TOKEN,
                auth_type="pat",
            ))
        else:
            thread_workspace_client = WorkspaceClient()

        mcp_client = DatabricksMCPClient(
            server_url=mcp_server_url,
            workspace_client=thread_workspace_client
        )

        result = mcp_client.call_tool(mcp_tool_name, args)

        if hasattr(result, 'content'):
            if isinstance(result.content, list):
                return "\n".join(
                    str(item.text if hasattr(item, 'text') else item)
                    for item in result.content
                )
            return str(result.content)
        return str(result)

    except BaseException as e:
        # Recursively unwrap TaskGroup / ExceptionGroup to surface the root cause
        real_error = e
        while hasattr(real_error, 'exceptions') and real_error.exceptions:
            real_error = real_error.exceptions[0]
        logger.error(f"MCP call failed for {tool_name}: {type(real_error).__name__}: {real_error}")
        raise real_error from None
    finally:
        loop.close()


########################################
# LAKEBASE CONNECTION POOLING
########################################

class CredentialConnection(psycopg.Connection):
    """Custom connection class with OAuth token caching."""

    workspace_client = None
    instance_name = None
    _cached_credential = None
    _cache_timestamp = None
    _cache_duration = 3000  # 50 minutes
    _cache_lock = Lock()

    @classmethod
    def connect(cls, conninfo='', **kwargs):
        if cls.workspace_client is None or cls.instance_name is None:
            raise ValueError("workspace_client and instance_name must be set")
        kwargs['password'] = cls._get_cached_credential()
        return super().connect(conninfo, **kwargs)

    @classmethod
    def _get_cached_credential(cls):
        with cls._cache_lock:
            current_time = time.time()
            if (cls._cached_credential is not None and
                cls._cache_timestamp is not None and
                current_time - cls._cache_timestamp < cls._cache_duration):
                return cls._cached_credential

            credential = cls.workspace_client.database.generate_database_credential(
                request_id=str(uuid.uuid4()),
                instance_names=[cls.instance_name]
            )
            cls._cached_credential = credential.token
            cls._cache_timestamp = current_time
            return cls._cached_credential


########################################
# GENIE CONFIGURATION
########################################

class GenieConfig(BaseModel):
    space_id: str
    name: str
    description: str = ""
    max_retries: int = 2


GENIE_CONFIG = GenieConfig(
    space_id=GENIE_SPACE_ID,
    name=GENIE_NAME,
    description="""This agent is a FALLBACK for when UC functions cannot answer the question.
The Genie space contains comprehensive baseball Statcast pitch-level data including:
- Pitch characteristics (velocity, spin, movement, release point)
- Batter outcomes (launch speed, angle, hit distance, wOBA)
- Player and team information
- Historical matchup data
Use this for flexible, novel queries that predefined UC functions cannot handle.

SQL generation guardrails:
- Generate Databricks SQL syntax only.
- For zipping arrays, use arrays_zip(...) with an s. Do not use array_zip(...).
- Prefer simple GROUP BY, CASE WHEN, avg, count, sum, and percentile calculations over complex array transformations.
- If exploding arrays is required, use posexplode or explode with valid Databricks SQL syntax.
- Query only tables attached to the Genie space.""",
    max_retries=2,
)


########################################
# AGENT STATE
########################################

class AgentState(TypedDict):
    """State for the combined agent with memory and Genie fallback."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    original_query: str
    uc_response: str | None
    uc_answered_parts: str | None
    unanswered_parts: str | None
    uc_sufficient: bool
    genie_response: str | None
    genie_error: str | None
    final_response: str | None
    tool_calls_made: list[dict[str, Any]]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]


########################################
# LOAD UC TOOLS
########################################

tools = []
if UC_TOOL_NAMES:
    try:
        print(f"[STARTUP] Loading {len(UC_TOOL_NAMES)} UC tools...")
        uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
        tools.extend(uc_toolkit.tools)
        print(f"[STARTUP] UC tools loaded successfully: {len(tools)} tools")
    except Exception as e:
        print(f"[STARTUP ERROR] Failed to load UC toolkit: {e}")
        logger.error(f"Failed to load UC toolkit: {e}")
else:
    print("[STARTUP] No UC tools configured")


# Build lookup for resolving potentially truncated tool names from LLM
_TOOL_NAME_SUFFIX_MAP = {}
for _tn in UC_TOOL_NAMES:
    # Full dotted name -> underscored MCP name
    _mcp_name = _tn.replace(".", "__")
    _TOOL_NAME_SUFFIX_MAP[_mcp_name] = _mcp_name
    # Also map by just the function name (last segment)
    _func_name = _tn.split(".")[-1]
    _TOOL_NAME_SUFFIX_MAP[_func_name] = _mcp_name


def _resolve_tool_name(name: str) -> str:
    """Resolve a potentially truncated tool name to the correct full MCP name.
    
    The LLM sometimes truncates long catalog names (e.g. 'cmegdemos_catalog' -> 'mos_catalog').
    This resolves by matching the function suffix.
    """
    if name in _TOOL_NAME_SUFFIX_MAP:
        return _TOOL_NAME_SUFFIX_MAP[name]
    # Try matching by suffix (function name after last __)
    parts = name.split("__")
    func_suffix = parts[-1] if parts else name
    if func_suffix in _TOOL_NAME_SUFFIX_MAP:
        resolved = _TOOL_NAME_SUFFIX_MAP[func_suffix]
        logger.info(f"Resolved truncated tool name '{name}' -> '{resolved}'")
        return resolved
    # Try endswith matching
    for full_name in _TOOL_NAME_SUFFIX_MAP.values():
        if full_name.endswith(func_suffix):
            logger.info(f"Resolved truncated tool name '{name}' -> '{full_name}'")
            return full_name
    return name


########################################
# MAIN AGENT CLASS
########################################

class CombinedLangGraphAgent(ResponsesAgent):
    """
    Combined agent with:
    - UC-first, Genie-fallback architecture
    - Parallel MCP tool execution
    - Lakebase PostgreSQL memory
    - Partial answer detection
    """

    def __init__(self, lakebase_config: dict[str, Any], genie_config: GenieConfig):
        self.lakebase_config = lakebase_config
        self.genie_config = genie_config
        self.workspace_client = WORKSPACE_CLIENT

        # LLM setup
        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

        # Genie agent
        self.genie_agent = GenieAgent(
            genie_space_id=genie_config.space_id,
            genie_agent_name=genie_config.name,
            description=genie_config.description,
        )

        # Connection pool settings
        self.pool_min_size = int(os.getenv("DB_POOL_MIN_SIZE", "1"))
        self.pool_max_size = int(os.getenv("DB_POOL_MAX_SIZE", "10"))
        self.pool_timeout = float(os.getenv("DB_POOL_TIMEOUT", "30.0"))

        cache_duration_minutes = int(os.getenv("DB_TOKEN_CACHE_MINUTES", "50"))
        CredentialConnection._cache_duration = cache_duration_minutes * 60

        # Only create pool if Lakebase is configured
        self._connection_pool = None
        if self.lakebase_config.get("conn_host"):
            try:
                self._connection_pool = self._create_rotating_pool()
            except Exception as e:
                logger.warning(f"Failed to create Lakebase connection pool: {e}")

        mlflow.langchain.autolog()

    def _get_username(self) -> str:
        try:
            sp = self.workspace_client.current_service_principal.me()
            return sp.application_id
        except Exception:
            user = self.workspace_client.current_user.me()
            return user.user_name

    def _create_rotating_pool(self) -> ConnectionPool:
        CredentialConnection.workspace_client = self.workspace_client
        CredentialConnection.instance_name = self.lakebase_config["instance_name"]

        username = self._get_username()
        host = self.lakebase_config["conn_host"]
        database = self.lakebase_config.get("conn_db_name", "databricks_postgres")

        pool = ConnectionPool(
            conninfo=f"dbname={database} user={username} host={host} sslmode=require options='-c search_path=checkpoint'",
            connection_class=CredentialConnection,
            min_size=self.pool_min_size,
            max_size=self.pool_max_size,
            timeout=self.pool_timeout,
            open=True,
            check=ConnectionPool.check_connection,
            kwargs={
                "autocommit": True,
                "row_factory": dict_row,
                "keepalives": 1,
                "keepalives_idle": 60,
                "keepalives_interval": 10,
                "keepalives_count": 5,
            }
        )

        # Test connection
        with pool.connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute("SELECT 1")

        return pool

    @contextmanager
    def get_connection(self):
        """Get a validated connection from the pool with retry logic."""
        if not self._connection_pool:
            yield None
            return

        max_retries = 3
        retry_count = 0

        while retry_count < max_retries:
            try:
                with self._connection_pool.connection() as conn:
                    with conn.cursor() as cursor:
                        cursor.execute("SELECT 1")
                    yield conn
                    return
            except (psycopg.OperationalError, psycopg.InterfaceError) as e:
                retry_count += 1
                if retry_count >= max_retries:
                    self._connection_pool = self._create_rotating_pool()
                    with self._connection_pool.connection() as conn:
                        yield conn
                        return
                time.sleep(0.5 * retry_count)

    def _parallel_tool_node(self, state: AgentState) -> dict:
        """Execute all tool calls in parallel using MCP with MLflow tracing."""
        messages = state["messages"]
        last_message = messages[-1] if messages else None

        if not isinstance(last_message, AIMessage) or not last_message.tool_calls:
            return {"messages": [], "tool_calls_made": []}

        tool_calls = last_message.tool_calls
        start_time = time.time()
        tool_results = []

        max_workers = int(os.getenv("UC_TOOL_MAX_WORKERS", "4"))
        with ThreadPoolExecutor(max_workers=min(len(tool_calls), max_workers)) as executor:
            futures = {}
            tool_start_times = {}
            for tc in tool_calls:
                tool_start_times[tc["id"]] = time.time()
                normalized_args, missing_required = _normalize_tool_args(tc["name"], tc.get("args"))
                if missing_required:
                    result = f"Error: Missing required tool parameter(s): {', '.join(missing_required)}"
                    tool_results.append({
                        "id": tc["id"],
                        "name": tc["name"],
                        "args": normalized_args,
                        "result": result,
                        "duration": 0.0,
                        "error": True,
                        "failure_type": "missing_required_parameter",
                    })
                    continue
                future = executor.submit(_execute_uc_function_with_retries, tc["name"], normalized_args)
                futures[future] = {"id": tc["id"], "name": tc["name"], "args": normalized_args}

            for future in as_completed(futures):
                tool_info = futures[future]
                elapsed_tool = time.time() - tool_start_times[tool_info["id"]]
                try:
                    result = future.result()
                    error = False
                    failure_type = None
                except Exception as e:
                    result = f"Error: {str(e)}"
                    error = True
                    failure_type = "transient_execution" if _is_transient_tool_error(e) else "execution_error"

                tool_results.append({
                    "id": tool_info["id"],
                    "name": tool_info["name"],
                    "args": tool_info["args"],
                    "result": result,
                    "duration": elapsed_tool,
                    "error": error,
                    "failure_type": failure_type,
                })

        elapsed = time.time() - start_time
        logger.info(f"Parallel tool execution: {len(tool_results)} tools in {elapsed:.2f}s")

        # MLflow tracing for parallel tool execution
        with mlflow.start_span(
            name=f"parallel_tools ({len(tool_results)} tools, {elapsed:.2f}s)",
            span_type=SpanType.TOOL
        ) as parent_span:
            parent_span.set_inputs({
                "num_tools": len(tool_results),
                "execution_mode": "parallel_mcp",
                "tool_names": [tr["name"] for tr in tool_results]
            })
            parent_span.set_attribute("wall_time_seconds", elapsed)

            for tr in tool_results:
                with mlflow.start_span(
                    name=f"{tr['name']} ({tr['duration']:.2f}s)",
                    span_type=SpanType.TOOL
                ) as span:
                    span.set_inputs({"tool_name": tr["name"], "args": tr["args"]})
                    span.set_outputs({
                        "execution_time_seconds": tr["duration"],
                        "result_preview": tr["result"][:500] if tr["result"] else "",
                        "error": tr["error"],
                        "failure_type": tr.get("failure_type"),
                    })

            parent_span.set_outputs({
                "completed": len(tool_results),
                "total_wall_time_seconds": elapsed,
                "failure_types": [tr.get("failure_type") for tr in tool_results if tr.get("failure_type")],
            })

        tool_messages = [
            ToolMessage(content=tr["result"], tool_call_id=tr["id"])
            for tr in tool_results
        ]

        return {"messages": tool_messages, "tool_calls_made": tool_results}

    def _summarize_tool_calls_for_eval(self, tool_calls_made: list[dict[str, Any]]) -> str:
        """Create a compact, judge-friendly summary of tool activity."""
        if not tool_calls_made:
            return "No tool calls were made."

        summaries = []
        for idx, tc in enumerate(tool_calls_made[:8], start=1):
            args = tc.get("args")
            if isinstance(args, dict):
                args_text = json.dumps(args, sort_keys=True)
            else:
                args_text = str(args)
            if len(args_text) > 300:
                args_text = args_text[:300] + "..."

            result_text = tc.get("result") or ""
            result_text = str(result_text)
            if len(result_text) > 600:
                result_text = result_text[:600] + "..."

            status = "error" if tc.get("error") else "ok"
            summaries.append(
                f"Tool {idx}: name={tc.get('name')} | status={status} | args={args_text} | output={result_text}"
            )

        if len(tool_calls_made) > 8:
            summaries.append(f"... {len(tool_calls_made) - 8} additional tool calls omitted")

        return "\n".join(summaries)

    def _format_tool_result_for_draft(self, tool_name: str, result: str) -> str:
        """Create a deterministic textual draft from common UC tool payloads."""
        try:
            payload = json.loads(result)
        except Exception:
            payload = None

        if not isinstance(payload, dict):
            return str(result)[:500]

        columns = payload.get("columns") or []
        rows = payload.get("rows") or []
        if not rows:
            return f"{tool_name} returned no rows."

        if columns == ["output"] and rows and rows[0]:
            nested_raw = rows[0][0]
            try:
                nested = json.loads(nested_raw)
            except Exception:
                nested = None

            if isinstance(nested, list) and nested:
                formatted = []
                for item in nested[:6]:
                    if not isinstance(item, dict):
                        formatted.append(str(item))
                        continue
                    pitch_name = item.get("pitch_name") or item.get("pitch_type") or "unknown pitch"
                    location = item.get("location_zone")
                    frequency = item.get("frequency_pct")
                    pitch_count = item.get("pitch_count")
                    parts = [str(pitch_name)]
                    if location:
                        parts.append(f"at {location}")
                    if frequency is not None:
                        parts.append(f"{frequency}%")
                    if pitch_count is not None:
                        parts.append(f"({pitch_count} pitches)")
                    formatted.append(" ".join(parts))
                return "; ".join(formatted)

        sample_rows = []
        for row in rows[:3]:
            if not isinstance(row, list):
                sample_rows.append(str(row))
                continue
            pairs = []
            for col, value in zip(columns, row):
                pairs.append(f"{col}={value}")
            sample_rows.append(", ".join(pairs))
        return " | ".join(sample_rows)

    def _draft_uc_response_from_tools(self, original_query: str, tool_calls_made: list[dict[str, Any]]) -> str:
        """Draft a direct answer from tool outputs when the model never wrote one."""
        if not tool_calls_made:
            return ""

        successful_calls = [tc for tc in tool_calls_made if not tc.get("error")]
        if not successful_calls:
            return ""

        drafted_parts = []
        for tc in successful_calls[:4]:
            formatted_result = self._format_tool_result_for_draft(tc.get("name", "tool"), str(tc.get("result") or ""))
            drafted_parts.append(formatted_result)

        joined = " ".join(part for part in drafted_parts if part).strip()
        if not joined:
            return ""

        return f"Based on the available tool outputs for '{original_query}', the retrieved evidence was: {joined}"

    def _evaluate_sufficiency(self, state: AgentState) -> dict:
        """Evaluate if UC response is sufficient, partial, or not answered."""
        original_query = state.get("original_query", "")
        uc_response = state.get("uc_response", "")
        tool_calls_made = state.get("tool_calls_made", [])
        tool_summary = self._summarize_tool_calls_for_eval(tool_calls_made)
        uc_response_source = "agent_response"

        if not (uc_response or "").strip() and tool_calls_made:
            uc_response = self._draft_uc_response_from_tools(original_query, tool_calls_made)
            if uc_response.strip():
                uc_response_source = "tool_draft"

        eval_prompt = _format_prompt_template(
            SUFFICIENCY_PROMPT,
            original_query=original_query,
            uc_response=uc_response or "[EMPTY RESPONSE]",
            tool_summary=tool_summary,
        )

        eval_result = self.model.invoke([HumanMessage(content=eval_prompt)])
        eval_text = _strip_reasoning_artifacts(eval_result.content)
        eval_content = eval_text.upper()

        is_sufficient = "FULLY_ANSWERED" in eval_content
        is_partial = "PARTIALLY_ANSWERED" in eval_content

        answered_parts = None
        unanswered_parts = None
        rationale = None

        try:
            lines = eval_text.split('\n')
            for line in lines:
                upper_line = line.upper()
                if upper_line.startswith("ANSWERED_PARTS:"):
                    answered_parts = line.split(":", 1)[1].strip()
                    if answered_parts.upper() == "NONE":
                        answered_parts = None
                elif upper_line.startswith("UNANSWERED_PARTS:"):
                    unanswered_parts = line.split(":", 1)[1].strip()
                    if unanswered_parts.upper() == "NONE":
                        unanswered_parts = None
                elif upper_line.startswith("RATIONALE:"):
                    rationale = line.split(":", 1)[1].strip()
        except Exception as e:
            logger.warning(f"Failed to parse sufficiency evaluation: {e}")

        if is_partial and unanswered_parts:
            status = "PARTIALLY_ANSWERED"
            result = {
                "uc_sufficient": False,
                "uc_answered_parts": answered_parts,
                "unanswered_parts": unanswered_parts,
                "uc_response": uc_response,
            }
        elif is_sufficient:
            status = "FULLY_ANSWERED"
            result = {
                "uc_sufficient": True,
                "uc_answered_parts": answered_parts,
                "unanswered_parts": None,
                "uc_response": uc_response,
            }
        else:
            status = "NOT_ANSWERED"
            result = {
                "uc_sufficient": False,
                "uc_answered_parts": None,
                "unanswered_parts": original_query,
                "uc_response": uc_response,
            }

        with mlflow.start_span(name=f"sufficiency_eval ({status})", span_type=SpanType.LLM) as span:
            span.set_inputs({
                "original_query": original_query,
                "uc_response": uc_response,
                "uc_response_source": uc_response_source,
                "tool_calls_summary": tool_summary,
                "num_tool_calls": len(tool_calls_made),
                "eval_prompt": eval_prompt,
            })
            span.set_outputs({
                "status": status,
                "answered_parts": answered_parts,
                "unanswered_parts": result["unanswered_parts"],
                "rationale": rationale,
                "will_fallback_to_genie": not result["uc_sufficient"],
                "raw_eval_text": eval_text,
            })

        return result

    def _genie_fallback_node(self, state: AgentState) -> dict:
        """Fall back to Genie for unanswered parts."""
        original_query = state.get("original_query", "")
        unanswered_parts = state.get("unanswered_parts", "")

        if unanswered_parts and unanswered_parts != original_query:
            genie_query = f"""I need help answering a specific part of a question about baseball data.
The user's full question was: {original_query}

I was able to answer part of it, but I need your help with:
{unanswered_parts}

Please focus on answering ONLY the part I couldn't answer."""
        else:
            genie_query = f"""I need help answering this question about baseball data.
Question: {original_query}
Please use your data access capabilities to answer this question."""

        genie_messages = [HumanMessage(content=genie_query)]

        genie_response = ""
        genie_error = None
        start_time = time.time()

        for attempt in range(self.genie_config.max_retries + 1):
            try:
                logger.info(f"Genie fallback attempt {attempt + 1}")
                result = self.genie_agent.invoke({"messages": genie_messages})

                for msg in reversed(result.get("messages", [])):
                    if hasattr(msg, "content") and msg.content:
                        genie_response = _strip_reasoning_artifacts(msg.content)
                        break

                if genie_response and genie_response.strip().upper() not in {"EMPTY", "NO DATA", "NO RESULTS"}:
                    break
                genie_response = ""

            except Exception as e:
                genie_error = f"Genie error: {str(e)}"
                logger.warning(f"Genie error on attempt {attempt + 1}: {e}")
                if attempt < self.genie_config.max_retries:
                    time.sleep(2 ** attempt)

        elapsed = time.time() - start_time

        with mlflow.start_span(name=f"genie_fallback ({elapsed:.2f}s)", span_type=SpanType.CHAIN) as span:
            span.set_inputs({"genie_space_id": self.genie_config.space_id, "original_query": original_query})
            span.set_outputs({"success": bool(genie_response), "duration_seconds": elapsed})

        return {"genie_response": genie_response, "genie_error": genie_error}

    def _build_fallback_action_plan(self, state: AgentState) -> str:
        """Return a useful terminal response when exact retrieval fails or is empty."""
        original_query = state.get("original_query", "")
        uc_response = state.get("uc_response", "")
        tool_calls_made = state.get("tool_calls_made", [])
        genie_error = state.get("genie_error")

        checked = []
        failures = []
        empty_results = []
        for tc in tool_calls_made[:8]:
            name = str(tc.get("name", "tool")).split("__")[-1]
            result_text = str(tc.get("result") or "")
            if tc.get("error"):
                failures.append(name)
            elif "rows" in result_text or result_text.strip():
                checked.append(name)
                if "\"rows\": []" in result_text or "returned no rows" in result_text.lower():
                    empty_results.append(name)

        checked_text = ", ".join(dict.fromkeys(checked)) or "the available baseball data tools"
        failure_text = ", ".join(dict.fromkeys(failures)) or None
        empty_text = ", ".join(dict.fromkeys(empty_results)) or None

        data_notes = [f"Checked {checked_text}."]
        if failure_text:
            data_notes.append(f"Some retrieval paths failed or timed out: {failure_text}.")
        if empty_text:
            data_notes.append(f"Some successful lookups returned no rows: {empty_text}.")
        if genie_error:
            data_notes.append(f"Genie fallback was unavailable: {genie_error}.")
        if uc_response:
            data_notes.append(f"Partial evidence: {uc_response[:500]}")

        return f"""# At-Bat Assistant Assessment
## Data collected
- {' '.join(data_notes)}

## Pitcher Approach
- The exact matchup slice was not fully available, so treat this as a preparation framework rather than a definitive scouting report. Anchor the plan on handedness, count leverage, pitch mix, location patterns, and runner state. Look for fastball counts early, protect against the primary secondary pitch with two strikes, and avoid expanding to chase zones until the pitcher proves he can land those pitches for strikes.

## Recommendation
- Use a selective, count-aware approach: hunt a pitch in one zone before two strikes, shrink the zone with runners on base, and force the pitcher to execute secondary stuff in the strike zone. If you need a precise recommendation, rerun with the pitcher, batter, season, count, handedness, and base state explicitly specified.
"""

    def _synthesize_response(self, state: AgentState) -> dict:
        """Synthesize final response from UC and Genie outputs."""
        uc_response = state.get("uc_response", "")
        genie_response = state.get("genie_response", "")
        genie_error = state.get("genie_error")
        uc_sufficient = state.get("uc_sufficient", False)
        uc_answered_parts = state.get("uc_answered_parts")
        unanswered_parts = state.get("unanswered_parts")
        original_query = state.get("original_query", "")

        if uc_sufficient:
            final_response = uc_response
        elif genie_response:
            if uc_answered_parts and unanswered_parts != original_query:
                synthesis_prompt = _format_prompt_template(
                    PARTIAL_SYNTHESIS_PROMPT,
                    original_query=original_query,
                    uc_response=uc_response,
                    genie_response=genie_response,
                )
            else:
                synthesis_prompt = _format_prompt_template(
                    SYNTHESIS_PROMPT,
                    original_query=original_query,
                    uc_response=uc_response,
                    genie_response=genie_response,
                )

            synthesis_result = self.model.invoke([HumanMessage(content=synthesis_prompt)])
            final_response = _strip_reasoning_artifacts(synthesis_result.content)

        elif genie_error:
            if uc_response and uc_answered_parts:
                final_response = f"""{uc_response}

---
{self._build_fallback_action_plan(state)}"""
            else:
                final_response = self._build_fallback_action_plan(state)
        else:
            final_response = uc_response or self._build_fallback_action_plan(state)

        final_response = _strip_reasoning_artifacts(final_response)

        return {"final_response": final_response, "messages": [AIMessage(content=final_response)]}

    def _create_graph(self, checkpointer=None):
        """Create the LangGraph workflow with UC-first, Genie-fallback pattern."""

        def should_continue_tools(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1] if messages else None
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "tools"
            return "evaluate"

        def route_after_evaluation(state: AgentState) -> Literal["genie_fallback", "synthesize"]:
            if state.get("uc_sufficient", False):
                return "synthesize"
            return "genie_fallback"

        # Preprocessor with system prompt
        if self.system_prompt:
            prompt_text = self.system_prompt.format() if hasattr(self.system_prompt, 'format') else str(self.system_prompt)
            preprocessor = RunnableLambda(
                lambda state: [{"role": "system", "content": prompt_text}] + list(state["messages"])
            )
        else:
            preprocessor = RunnableLambda(lambda state: list(state["messages"]))

        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            if isinstance(response, AIMessage) and response.content:
                response.content = _strip_reasoning_artifacts(response.content)
            original_query = state.get("original_query", "")
            if not original_query:
                for msg in reversed(state["messages"]):
                    if isinstance(msg, HumanMessage):
                        original_query = msg.content
                        break
            return {"messages": [response], "original_query": original_query}

        def extract_uc_response(state: AgentState):
            messages = state["messages"]
            uc_response = ""
            for msg in reversed(messages):
                if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
                    uc_response = _strip_reasoning_artifacts(msg.content)
                    break
            return {"uc_response": uc_response}

        workflow = StateGraph(AgentState)

        # Add nodes
        workflow.add_node("agent", RunnableLambda(call_model))
        workflow.add_node("tools", self._parallel_tool_node)
        workflow.add_node("extract_response", extract_uc_response)
        workflow.add_node("evaluate", self._evaluate_sufficiency)
        workflow.add_node("genie_fallback", self._genie_fallback_node)
        workflow.add_node("synthesize", self._synthesize_response)

        # Add edges
        workflow.set_entry_point("agent")
        workflow.add_conditional_edges(
            "agent",
            should_continue_tools,
            {"tools": "tools", "evaluate": "extract_response"}
        )
        workflow.add_edge("tools", "agent")
        workflow.add_edge("extract_response", "evaluate")
        workflow.add_conditional_edges(
            "evaluate",
            route_after_evaluation,
            {"genie_fallback": "genie_fallback", "synthesize": "synthesize"}
        )
        workflow.add_edge("genie_fallback", "synthesize")
        workflow.add_edge("synthesize", END)

        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        ci = dict(request.custom_inputs or {})
        if "thread_id" in ci:
            return ci["thread_id"]
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id
        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs={"thread_id": thread_id})

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Streaming prediction with PostgreSQL checkpointing and MLflow tracing."""
        thread_id = self._get_or_create_thread_id(request)

        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        if os.getenv("ATBAT_MLFLOW_LOG_MODEL_DRY_RUN", "false").lower() == "true":
            yield ResponsesAgentStreamEvent(
                type="response.output_item.done",
                item=self.create_text_output_item(
                    text="Model logging validation response. Live model calls are skipped during packaging.",
                    id=str(uuid.uuid4()),
                ),
                custom_outputs={"thread_id": thread_id},
            )
            return

        cc_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        checkpoint_config = {
            "configurable": {"thread_id": thread_id},
            "recursion_limit": int(os.getenv("LANGGRAPH_RECURSION_LIMIT", "25")),
        }

        # Convert to LangChain messages
        lc_messages = []
        for msg in cc_msgs:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            if role == "user":
                lc_messages.append(HumanMessage(content=content))
            elif role == "assistant":
                lc_messages.append(AIMessage(content=content))

        # MLflow tracing for Lakebase session
        with mlflow.start_span(name="lakebase_session", span_type=SpanType.RETRIEVER) as db_span:
            db_span.set_inputs({
                "thread_id": thread_id,
                "lakebase_instance": self.lakebase_config.get("instance_name", "unknown"),
                "memory_enabled": bool(self._connection_pool)
            })

            with self.get_connection() as conn:
                checkpointer = PostgresSaver(conn) if conn else None
                graph = self._create_graph(checkpointer=checkpointer)

                node_count = 0
                final_response = None
                emitted_tool_calls = set()
                tracked_unanswered_parts = ""

                for event in graph.stream(
                    {"messages": lc_messages},
                    checkpoint_config,
                    stream_mode=["updates", "messages"]
                ):
                    if event[0] == "updates":
                        node_count += 1
                        node_name = list(event[1].keys())[0] if event[1] else ""
                        node_data = event[1].get(node_name, {})

                        # Emit function_call events when tools node executes
                        if node_name == "tools" and "tool_calls_made" in node_data:
                            for tc in node_data["tool_calls_made"]:
                                if tc["id"] not in emitted_tool_calls:
                                    emitted_tool_calls.add(tc["id"])
                                    yield ResponsesAgentStreamEvent(
                                        type="response.output_item.done",
                                        item=self.create_function_call_item(
                                            id=str(uuid.uuid4()),
                                            call_id=tc["id"],
                                            name=tc["name"],
                                            arguments=json.dumps(tc["args"]) if isinstance(tc["args"], dict) else tc["args"],
                                        ),
                                        custom_outputs={"thread_id": thread_id}
                                    )
                                    yield ResponsesAgentStreamEvent(
                                        type="response.output_item.done",
                                        item=self.create_function_call_output_item(
                                            call_id=tc["id"],
                                            output=tc["result"][:2000] if tc["result"] else "",
                                        ),
                                        custom_outputs={"thread_id": thread_id}
                                    )

                        if node_name == "evaluate" and "unanswered_parts" in node_data:
                            tracked_unanswered_parts = node_data.get("unanswered_parts", "")

                        # Emit Genie fallback event
                        if node_name == "genie_fallback":
                            genie_call_id = f"genie-{uuid.uuid4()}"
                            genie_response = node_data.get("genie_response", "")
                            query_display = tracked_unanswered_parts if tracked_unanswered_parts else "Genie fallback query"
                            yield ResponsesAgentStreamEvent(
                                type="response.output_item.done",
                                item=self.create_function_call_item(
                                    id=str(uuid.uuid4()),
                                    call_id=genie_call_id,
                                    name="genie_space_query",
                                    arguments=json.dumps({"query": query_display[:200]}),
                                ),
                                custom_outputs={"thread_id": thread_id}
                            )
                            yield ResponsesAgentStreamEvent(
                                type="response.output_item.done",
                                item=self.create_function_call_output_item(
                                    call_id=genie_call_id,
                                    output=genie_response[:500] if genie_response else "(queried Genie space)",
                                ),
                                custom_outputs={"thread_id": thread_id}
                            )

                        # Track final response from synthesize node
                        if node_name == "synthesize" and "final_response" in node_data:
                            final_response = node_data["final_response"]
                        elif node_name == "synthesize" and "messages" in node_data:
                            for msg in node_data["messages"]:
                                if isinstance(msg, AIMessage) and msg.content:
                                    final_response = msg.content

                # Emit final synthesized response
                if final_response:
                    yield ResponsesAgentStreamEvent(
                        type="response.output_item.done",
                        item=self.create_text_output_item(
                            text=final_response,
                            id=str(uuid.uuid4()),
                        ),
                        custom_outputs={"thread_id": thread_id}
                    )

                db_span.set_outputs({
                    "status": "completed",
                    "thread_id": thread_id,
                    "nodes_executed": node_count,
                    "tool_calls_emitted": len(emitted_tool_calls)
                })


########################################
# INSTANTIATE AGENT
########################################

AGENT = CombinedLangGraphAgent(LAKEBASE_CONFIG, GENIE_CONFIG)
mlflow.models.set_model(AGENT)


In [ ]:
dbutils.library.restartPython()

## Test the Agent

### Expected Behavior
1. **UC Functions First**: The agent always tries to answer using Unity Catalog functions first
2. **Sufficiency Check**: LLM evaluates if the UC response is FULLY, PARTIALLY, or NOT answered
3. **Partial Answer Detection**: If UC answered some parts but not others, only unanswered parts go to Genie
4. **Genie Fallback**: For unanswered parts, Genie is automatically invoked
5. **Response Synthesis**: The final response intelligently combines both sources

In [ ]:
# Reload config after Python restart
import json
import os
from pathlib import Path
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Re-extract all configuration variables (needed for logging/deployment cells)
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
FINAL_SYNTHESIS_ENDPOINT_NAME = os.getenv(
    "ATBAT_FINAL_SYNTHESIS_ENDPOINT",
    CONFIG["llm"].get("final_synthesis_endpoint_name", "gpt-5-4-external"),
)
FINAL_SYNTHESIS_ENDPOINT_NAME = str(FINAL_SYNTHESIS_ENDPOINT_NAME or "").replace("databricks:/", "")
UC_MODEL_NAME = CONFIG["model"]["uc_model_name"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
DISABLE_VECTOR_TOOLS = os.getenv("DISABLE_VECTOR_TOOLS", "true").lower() in {"1", "true", "yes"}
if DISABLE_VECTOR_TOOLS:
    UC_TOOL_NAMES = [
        tool_name for tool_name in UC_TOOL_NAMES
        if "embedding" not in tool_name.lower() and "vector" not in tool_name.lower()
    ]
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]
LAKEBASE_INSTANCE = CONFIG["lakebase"]["instance_name"]

# Set MLflow experiment
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)


In [ ]:
import os
from agent import AGENT

RUN_LIVE_AGENT_TESTS = os.getenv("RUN_LIVE_AGENT_TESTS", "false").lower() == "true"

if RUN_LIVE_AGENT_TESTS:
    # Test 1: UC Functions Only
    print("=== Test 1: UC Functions Only ===")
    result = AGENT.predict({"input": [{"role": "user", "content": "How will Kyle Freeland pitch to Freddie Freeman?"}]})
    print(f"Thread ID: {result.custom_outputs.get('thread_id')}")
    print(f"Response preview: {result.output[-1].content[0]['text'][:500]}...")
else:
    print("Skipping live agent tests. Set RUN_LIVE_AGENT_TESTS=true to enable them.")


In [ ]:
# Test 2: UC Functions AND Genie
if RUN_LIVE_AGENT_TESTS:
    print("=== Test 2: UC Functions + Genie Fallback ===")
    result = AGENT.predict({"input": [{"role": "user", "content": "How will Blake Snell pitch to Mookie Betts? What was the full pitch distribution for the Rockies in 2025 including the average spin rate and velocity by pitch?"}]})
    print(f"Thread ID: {result.custom_outputs.get('thread_id')}")
    print(f"Response preview: {result.output[-1].content[0]['text'][:500]}...")
else:
    print("Skipping live Genie fallback test.")


In [ ]:
# Test 3: Streaming
if RUN_LIVE_AGENT_TESTS:
    print("=== Test 3: Streaming ===")
    for chunk in AGENT.predict_stream(
        {"input": [{"role": "user", "content": "How does Yu Darvish pitch to lefties with runners on 2nd?"}]}
    ):
        print(chunk.model_dump(exclude_none=True))
else:
    print("Skipping live streaming test.")


In [ ]:
# Test 4: Memory test (uses same thread_id)
if RUN_LIVE_AGENT_TESTS:
    print("=== Test 4: Memory Test ===")
    thread_id = "atbat-memory-test-001"

    result1 = AGENT.predict({
        "input": [{"role": "user", "content": "The demo memory phrase is inside fastball"}],
        "custom_inputs": {"thread_id": thread_id}
    })

    result2 = AGENT.predict({
        "input": [{"role": "user", "content": "What demo memory phrase did I provide?"}],
        "custom_inputs": {"thread_id": thread_id}
    })
    print(f"Memory response: {result2.output[-1].content[0]['text'][:300]}")
else:
    print("Skipping memory test because Lakebase is disabled in Free Edition.")


## Log and Deploy the Agent

In [ ]:
import os

# Determine Databricks resources for automatic auth passthrough at deployment time.
# Declare only resources that are configured and available in this workspace.
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
    DatabricksTable,
)
from pkg_resources import get_distribution


def _is_configured(value):
    return bool(value) and not str(value).startswith("PLACEHOLDER_")


warehouse_id = CONFIG["genie"].get("warehouse_id", "")
genie_space_id = CONFIG["genie"].get("space_id", "")
uc_schema = f"{CATALOG}.{SCHEMA}"

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
final_synthesis_endpoint_name = FINAL_SYNTHESIS_ENDPOINT_NAME
if final_synthesis_endpoint_name and final_synthesis_endpoint_name != LLM_ENDPOINT_NAME:
    resources.append(DatabricksServingEndpoint(endpoint_name=final_synthesis_endpoint_name))

for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))

if _is_configured(warehouse_id):
    resources.append(DatabricksSQLWarehouse(warehouse_id=warehouse_id))
else:
    print("Skipping SQL warehouse resource because no warehouse is configured.")

if _is_configured(genie_space_id):
    resources.append(DatabricksGenieSpace(genie_space_id=genie_space_id))
    for table in CONFIG["genie"].get("tables", []):
        resources.append(DatabricksTable(table_name=f"{uc_schema}.{table}"))
else:
    print("Skipping Genie resources because no Genie space is configured.")

print("Skipping Lakebase resource because Free Edition does not support Lakebase database instances.")

input_example = {
    "input": [
        {"role": "user", "content": "How does Kyle Freeland pitch to Freddie Freeman?"}
    ]
}

os.environ["ATBAT_MLFLOW_LOG_MODEL_DRY_RUN"] = "true"
try:
    with mlflow.start_run():
        logged_agent_info = mlflow.pyfunc.log_model(
            name="agent",
            python_model="agent.py",
            input_example=input_example,
            resources=resources,
            pip_requirements=[
                "databricks-openai",
                "backoff",
                f"databricks-connect=={get_distribution('databricks-connect').version}",
                f"databricks-langchain=={get_distribution('databricks-langchain').version}",
                f"langgraph=={get_distribution('langgraph').version}",
                "langgraph-checkpoint-postgres",
                "psycopg[binary,pool]",
                "databricks-mcp",
            ],
        )
        print(f"Logged model: {logged_agent_info.model_uri}")
finally:
    os.environ.pop("ATBAT_MLFLOW_LOG_MODEL_DRY_RUN", None)


In [ ]:
# Register to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri,
    name=UC_MODEL_NAME
)
print(f"Registered model: {UC_MODEL_NAME} version {uc_registered_model_info.version}")

In [ ]:
# Deploy the agent
from databricks import agents

# Build environment variables. Package the config so deployed agent can load it.
config_payload = Path("config/atbat_assistant.json").read_text()
environment_vars = {
    "ATBAT_ASSISTANT_CONFIG_JSON": config_payload,
    "MLFLOW_TRACKING_URI": "databricks",
}

# Build authentication environment variables
auth_config = CONFIG["prompt_registry_auth"]
if auth_config.get("databricks_host"):
    if auth_config.get("use_oauth", True):
        environment_vars.update({
            "DATABRICKS_HOST": auth_config["databricks_host"],
            "DATABRICKS_CLIENT_ID": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config['oauth_client_id_key']}}}}}",
            "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config['oauth_client_secret_key']}}}}}",
        })
        print(f"Using OAuth authentication (scope: {auth_config['secret_scope_name']})")
        print(f"MLflow tracking URI set to 'databricks'. Traces will go to experiment {CONFIG['mlflow']['experiment_id']}")
    else:
        environment_vars.update({
            "DATABRICKS_HOST": auth_config["databricks_host"],
            "DATABRICKS_TOKEN": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config.get('pat_key', 'pat')}}}}}",
        })
        print(f"Using PAT authentication (scope: {auth_config['secret_scope_name']})")
else:
    print("WARNING: DATABRICKS_HOST not configured. Prompt Registry and MLflow tracking may not work.")

environment_vars["DISABLE_LAKEBASE"] = "true"
environment_vars["DISABLE_VECTOR_TOOLS"] = "true"
environment_vars["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "300"
environment_vars["MLFLOW_HTTP_REQUEST_MAX_RETRIES"] = "1"
print("Lakebase disabled for deployment because Free Edition does not support Lakebase database instances.")

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    scale_to_zero=True,
    environment_vars=environment_vars,
)

endpoint_name = f"agents_{UC_MODEL_NAME.replace('.', '-')}"
print(f"\nServing endpoint: {endpoint_name}")
print("Set this in your Databricks App config:")
print(f"  UC_MODEL_NAME={UC_MODEL_NAME}")
print("  -- or --")
print(f"  SERVING_ENDPOINT_NAME={endpoint_name}")


In [ ]:
# Grant CAN_QUERY on the serving endpoint to all workspace users.
# This ensures Databricks Apps can query the endpoint without a manual permission step.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint_name = f"agents_{UC_MODEL_NAME.replace('.', '-')}"
endpoint_info = w.serving_endpoints.get(endpoint_name)

w.api_client.do(
    "PUT",
    f"/api/2.0/permissions/serving-endpoints/{endpoint_info.id}",
    body={
        "access_control_list": [
            {
                "group_name": "users",
                "permission_level": "CAN_QUERY",
            }
        ]
    },
)
print(f"Granted CAN_QUERY on '{endpoint_name}' to all workspace users")
